In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 11
# What the Warning Owes: Measuring the Engineer-Person Asymmetry
# =============================================================
#
# This notebook does NOT train anything. It reads the three
# calibrated mechanisms from Essay #10's checkpoints and measures
# the asymmetry between:
#
#   (engineer's view)  what the gate's reliability actually was
#   (person's view)    the deferral rate the person experienced
#
# across four conditions and four attack budgets.
#
# Two inversion criteria are reported:
#
#   A. Binary: signal-opposite-to-reality
#      deferral_attack < deferral_clean
#      AND wrong_undeferred_attack > wrong_undeferred_clean
#
#   B. Magnitude (pp):
#      Δdeferral = deferral_clean − deferral_attack
#      Δexposure = wrong_undeferred_attack − wrong_undeferred_clean
#      inversion_magnitude = min(Δdeferral, Δexposure)
#        if both > 0 else 0
# =============================================================

In [ ]:
# -------------------------------------------------------------
# Part A: Imports and Setup
# -------------------------------------------------------------

import json
import math
import random
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

CHECKPOINT_DIR = Path("essay10_checkpoints")
RESULTS_DIR = Path("essay11_results")
RESULTS_DIR.mkdir(exist_ok=True)

MANIFEST_PATH = CHECKPOINT_DIR / "manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing manifest at {MANIFEST_PATH}.\n"
        "Run Part Y of the Essay #10 notebook first. It writes "
        "essay10_checkpoints/manifest.json."
    )

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    MANIFEST = json.load(f)

SEED = MANIFEST["seed"]
NUM_CLASSES = MANIFEST["num_classes"]
EPSILON = MANIFEST["epsilon"]
PGD_STEPS = MANIFEST["pgd_steps"]
TARGET_DEFERRAL = MANIFEST["target_deferral"]
CALIB_SIZE = MANIFEST["calibration_size"]
ATTACK_DEV_SIZE = MANIFEST["attack_dev_size"]
EVAL_SIZE = MANIFEST["evaluation_size"]
MC_EVAL_PASSES = MANIFEST["mc_eval_passes"]
MC_ATTACK_PASSES = MANIFEST["mc_attack_passes"]
CALIBRATION = MANIFEST["calibration"]

print(f"Loaded manifest from {MANIFEST_PATH}")
print(f"  seed={SEED}, ε={EPSILON:.6f} ({EPSILON*255:.1f}/255), "
      f"PGD steps={PGD_STEPS}")
print(f"  calibration target: {TARGET_DEFERRAL*100:.0f}% clean deferral")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


In [ ]:
# -------------------------------------------------------------
# Part B: Dataset Splits (must match #10 exactly)
# -------------------------------------------------------------

transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

split_generator = torch.Generator().manual_seed(SEED)

calibration_dataset, attack_dev_dataset, evaluation_dataset = random_split(
    test_dataset,
    [CALIB_SIZE, ATTACK_DEV_SIZE, EVAL_SIZE],
    generator=split_generator,
)

eval_loader = DataLoader(
    evaluation_dataset, batch_size=128, shuffle=False, num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print(f"Splits: calibration={len(calibration_dataset)}, "
      f"attack_dev={len(attack_dev_dataset)}, "
      f"evaluation={len(evaluation_dataset)}")


In [ ]:
# -------------------------------------------------------------
# Part C: Reconstruct the Three Models (must match #10's classes)
# -------------------------------------------------------------

CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], dtype=torch.float32).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010], dtype=torch.float32).view(1, 3, 1, 1)


def normalize_images(x: torch.Tensor) -> torch.Tensor:
    return (x - CIFAR_MEAN.to(x.device, x.dtype)) / CIFAR_STD.to(x.device, x.dtype)


class FeatureBackbone(nn.Module):
    def __init__(self, dropout_probability: float = 0.0):
        super().__init__()
        self.dropout_probability = dropout_probability
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.projection = nn.Linear(256, 128)

    def forward(self, x, *, stochastic_dropout: bool = False):
        x = normalize_images(x)
        x = self.block1(x)
        x = F.dropout2d(x, p=self.dropout_probability, training=stochastic_dropout)
        x = self.block2(x)
        x = F.dropout2d(x, p=self.dropout_probability, training=stochastic_dropout)
        x = self.block3(x)
        x = torch.flatten(x, 1)
        x = self.projection(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_probability, training=stochastic_dropout)
        return x


class MCDropoutCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = FeatureBackbone(dropout_probability=0.30)
        self.classifier = nn.Linear(128, NUM_CLASSES)

    def forward(self, x, *, stochastic: bool = False):
        return self.classifier(self.backbone(x, stochastic_dropout=stochastic))


class EvidentialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = FeatureBackbone(dropout_probability=0.0)
        self.evidence_head = nn.Linear(128, NUM_CLASSES)

    def forward(self, x):
        features = self.backbone(x)
        evidence = F.softplus(self.evidence_head(features))
        return evidence + 1.0


class GeometryCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = FeatureBackbone(dropout_probability=0.0)
        self.classifier = nn.Linear(128, NUM_CLASSES)

    def forward(self, x, *, return_features: bool = False):
        features = self.backbone(x)
        logits = self.classifier(features)
        return (logits, features) if return_features else logits


def load_checkpoint(model: nn.Module, filename: str) -> nn.Module:
    path = CHECKPOINT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}. Re-run #10 training.")
    state = torch.load(path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model.eval()
    return model.to(DEVICE)


mc_model = load_checkpoint(MCDropoutCNN(), MANIFEST["checkpoints"]["mc_dropout"])
evidential_model = load_checkpoint(EvidentialCNN(), MANIFEST["checkpoints"]["evidential"])
geometry_model = load_checkpoint(GeometryCNN(), MANIFEST["checkpoints"]["geometry"])

centroid_path = CHECKPOINT_DIR / MANIFEST["centroids"]
if not centroid_path.exists():
    raise FileNotFoundError(f"Missing centroids: {centroid_path}")
class_centroids = torch.load(centroid_path, map_location=DEVICE, weights_only=True)

print("Loaded all three models and centroids.")


In [ ]:
# -------------------------------------------------------------
# Part D: Uncertainty Signals and Predictions (match #10 exactly)
# -------------------------------------------------------------

def mc_predictive_distribution(model, images, passes=MC_EVAL_PASSES):
    samples = torch.stack(
        [F.softmax(model(images, stochastic=True), dim=1) for _ in range(passes)],
        dim=0,
    )
    mean_probs = samples.mean(dim=0)
    uncertainty = samples.var(dim=0, unbiased=False).max(dim=1).values
    return mean_probs, uncertainty


def evidential_predictive_distribution(model, images):
    alpha = model(images)
    strength = alpha.sum(dim=1, keepdim=True)
    return alpha / strength, NUM_CLASSES / strength.squeeze(1)


def geometry_predictive_distribution(model, images, centroids):
    logits, features = model(images, return_features=True)
    probs = F.softmax(logits, dim=1)
    norm_feats = F.normalize(features, p=2, dim=1)
    sq_dists = torch.cdist(norm_feats, centroids, p=2).pow(2)
    return probs, sq_dists.min(dim=1).values


def predict(model_kind, model, images, mc_passes=MC_EVAL_PASSES):
    if model_kind == "mc_dropout":
        return mc_predictive_distribution(model, images, passes=mc_passes)
    if model_kind == "evidential":
        return evidential_predictive_distribution(model, images)
    if model_kind == "geometry":
        return geometry_predictive_distribution(model, images, class_centroids)
    raise ValueError(model_kind)


In [ ]:
# -------------------------------------------------------------
# Part E: PGD Attacks (match #10 exactly)
# -------------------------------------------------------------

def project_linf(candidate, original, epsilon):
    perturbation = torch.clamp(candidate - original, -epsilon, epsilon)
    return torch.clamp(original + perturbation, 0.0, 1.0)


def geometry_wrong_class_distance(model, images, labels, centroids):
    logits, features = model(images, return_features=True)
    probs = F.softmax(logits, dim=1)
    norm_feats = F.normalize(features, p=2, dim=1)
    sq_dists = torch.cdist(norm_feats, centroids, p=2).pow(2)
    true_mask = F.one_hot(labels, NUM_CLASSES).bool()
    wrong_dists = sq_dists.masked_fill(true_mask, float("inf"))
    return probs, wrong_dists.min(dim=1).values


def adaptive_pgd(model_kind, model, images, labels, *, lam, scale,
                 epsilon=EPSILON, step_size=None, steps=PGD_STEPS):
    step_size = step_size or epsilon / 4.0
    model.eval()
    original = images.detach()
    adv = original.clone()
    scale = max(scale, 1e-8)

    for _ in range(steps):
        adv.requires_grad_(True)

        if model_kind == "geometry":
            probs, suppression = geometry_wrong_class_distance(model, adv, labels, class_centroids)
        else:
            probs, suppression = predict(model_kind, model, adv, mc_passes=MC_ATTACK_PASSES)

        cls_loss = F.nll_loss(torch.log(probs.clamp_min(1e-10)), labels)
        normalized = (suppression / scale).mean()
        objective = cls_loss - lam * normalized

        grad = torch.autograd.grad(objective, adv, only_inputs=True)[0]
        candidate = adv.detach() + step_size * grad.sign()
        adv = project_linf(candidate, original, epsilon).detach()
    return adv


def direct_pgd(model_kind, model, images, labels, *,
               epsilon=EPSILON, step_size=None, steps=PGD_STEPS):
    step_size = step_size or epsilon / 4.0
    model.eval()
    original = images.detach()
    adv = original.clone()
    for _ in range(steps):
        adv.requires_grad_(True)
        probs, _ = predict(model_kind, model, adv, mc_passes=MC_ATTACK_PASSES)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-10)), labels)
        grad = torch.autograd.grad(loss, adv, only_inputs=True)[0]
        candidate = adv.detach() + step_size * grad.sign()
        adv = project_linf(candidate, original, epsilon).detach()
    return adv


In [ ]:
# -------------------------------------------------------------
# Part F: Evaluation (asymmetry + two inversion criteria)
# -------------------------------------------------------------

def evaluate_condition(model_kind, model, loader, *, threshold, condition,
                       adaptive_lambda=None, uncertainty_scale=None,
                       attack_steps=PGD_STEPS, epsilon=EPSILON,
                       max_samples=None):
    model.eval()
    total = answered = deferred = correct_answered = wrong_undeferred = correct_total = 0
    unc_sum = 0.0
    n = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if condition == "clean":
            evaluated = images
        elif condition == "direct":
            evaluated = direct_pgd(model_kind, model, images, labels,
                                   epsilon=epsilon, steps=attack_steps)
        elif condition == "adaptive":
            evaluated = adaptive_pgd(model_kind, model, images, labels,
                                     lam=adaptive_lambda, scale=uncertainty_scale,
                                     epsilon=epsilon, steps=attack_steps)
        else:
            raise ValueError(condition)

        with torch.no_grad():
            probs, unc = predict(model_kind, model, evaluated)
        preds = probs.argmax(dim=1)
        correct = preds.eq(labels)
        defer = unc > threshold

        total += labels.size(0)
        correct_total += correct.sum().item()
        deferred += defer.sum().item()
        answered += (~defer).sum().item()
        correct_answered += (correct & ~defer).sum().item()
        wrong_undeferred += ((~correct) & ~defer).sum().item()
        unc_sum += unc.sum().item()

        if max_samples is not None:
            n += labels.size(0)
            if n >= max_samples:
                break

    acc_answered = 100.0 * correct_answered / answered if answered else float("nan")
    return {
        "n": total,
        "accuracy": 100.0 * correct_total / total,
        "coverage": 100.0 * answered / total,
        "deferral_rate": 100.0 * deferred / total,
        "risk_on_answered": 100.0 - acc_answered if answered else float("nan"),
        "wrong_undeferred": 100.0 * wrong_undeferred / total,
        "mean_uncertainty": unc_sum / total,
    }


In [ ]:
# -------------------------------------------------------------
# Part G: The Asymmetry and Inversion Measures
# -------------------------------------------------------------

def asymmetry(clean_def, attack_def):
    """
    Engineer's view vs person's view.

    Positive asymmetry: the person sees MORE deferral than clean,
    i.e., the gate looks MORE cautious under attack than it did
    when it was reliable. The person is being given a reassuring
    signal while the engineer knows the gate is compromised.

    Negative asymmetry: the person sees LESS deferral, i.e., the
    gate looks MORE confident under attack. The person is being
    given a *false* confidence signal.
    """
    return attack_def - clean_def


def inversion_binary(clean_def, clean_wu, attack_def, attack_wu):
    """
    Criterion A: binary inversion.

    True iff the person's signal moved opposite to reality:
      deferral fell  AND  exposure rose.
    """
    return (attack_def < clean_def) and (attack_wu > clean_wu)


def inversion_magnitude(clean_def, clean_wu, attack_def, attack_wu):
    """
    Criterion B: magnitude, in percentage points.

    Δdeferral = clean_def - attack_def   (>0 means deferral fell)
    Δexposure = attack_wu - clean_wu     (>0 means exposure rose)

    Magnitude = min(Δdeferral, Δexposure) if both > 0 else 0.
    """
    d_def = clean_def - attack_def
    d_exp = attack_wu - clean_wu
    if d_def > 0 and d_exp > 0:
        return min(d_def, d_exp), d_def, d_exp
    return 0.0, d_def, d_exp


In [ ]:
# -------------------------------------------------------------
# Part H: Run All Mechanisms, All Conditions
# -------------------------------------------------------------

models = {
    "mc_dropout": mc_model,
    "evidential": evidential_model,
    "geometry": geometry_model,
}

CONDITIONS = ["clean", "direct", "adaptive"]

# Reuse the adaptive lambdas selected in #10
SELECTED_LAMBDAS_PATH = Path("essay10_results_corrected/selected_lambdas.json")
if not SELECTED_LAMBDAS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SELECTED_LAMBDAS_PATH}. Re-run #10 through Part O."
    )
with open(SELECTED_LAMBDAS_PATH, "r", encoding="utf-8") as f:
    SELECTED_LAMBDAS = json.load(f)

print("\nSelected lambdas (from #10):", SELECTED_LAMBDAS)

rows = []
for mechanism, model in models.items():
    thr = CALIBRATION[mechanism]["threshold"]
    scale = CALIBRATION[mechanism]["scale"]
    lam = SELECTED_LAMBDAS[mechanism]

    print(f"\n=== {mechanism} ===")
    for condition in CONDITIONS:
        result = evaluate_condition(
            mechanism, model, eval_loader,
            threshold=thr, condition=condition,
            adaptive_lambda=lam if condition == "adaptive" else None,
            uncertainty_scale=scale if condition == "adaptive" else None,
        )
        rows.append({"mechanism": mechanism, "condition": condition, **result})
        print(
            f"  {condition:<9} "
            f"acc={result['accuracy']:6.2f}%  "
            f"def={result['deferral_rate']:6.2f}%  "
            f"WU={result['wrong_undeferred']:6.2f}%"
        )

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "per_condition.csv", index=False)


In [ ]:
# -------------------------------------------------------------
# Part I: The Asymmetry Table
# -------------------------------------------------------------

asym_rows = []
for mechanism in models:
    m = df[df["mechanism"] == mechanism].set_index("condition")
    clean = m.loc["clean"]
    for attack in ("direct", "adaptive"):
        a = m.loc[attack]
        a_def = asymmetry(clean["deferral_rate"], a["deferral_rate"])
        bin_inv = inversion_binary(
            clean["deferral_rate"], clean["wrong_undeferred"],
            a["deferral_rate"], a["wrong_undeferred"],
        )
        mag, d_def, d_exp = inversion_magnitude(
            clean["deferral_rate"], clean["wrong_undeferred"],
            a["deferral_rate"], a["wrong_undeferred"],
        )
        asym_rows.append({
            "mechanism": mechanism,
            "condition": attack,
            "clean_deferral": clean["deferral_rate"],
            "attack_deferral": a["deferral_rate"],
            "asymmetry_pp": a_def,
            "clean_wu": clean["wrong_undeferred"],
            "attack_wu": a["wrong_undeferred"],
            "delta_deferral_pp": d_def,
            "delta_exposure_pp": d_exp,
            "inversion_binary": bin_inv,
            "inversion_magnitude_pp": mag,
        })

asym_df = pd.DataFrame(asym_rows)
asym_df.to_csv(RESULTS_DIR / "asymmetry.csv", index=False)
print("\n" + "=" * 90)
print("ASYMMETRY AND INVERSION (per mechanism, direct and adaptive)")
print("=" * 90)
print(asym_df.to_string(index=False, float_format=lambda v: f"{v:.2f}"))


In [ ]:
# -------------------------------------------------------------
# Part J: Figure — Engineer's view vs person's view
# -------------------------------------------------------------

colors_clean = "#4C78A8"
colors_direct = "#E45756"
colors_adaptive = "#72B7B2"

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, mechanism in zip(axes, models):
    m = df[df["mechanism"] == mechanism].set_index("condition")
    conditions = ["clean", "direct", "adaptive"]
    x = np.arange(len(conditions))

    wu = [m.loc[c, "wrong_undeferred"] for c in conditions]
    defr = [m.loc[c, "deferral_rate"] for c in conditions]

    ax2 = ax.twinx()
    bars = ax.bar(x, wu, color=[colors_clean, colors_direct, colors_adaptive], alpha=0.85, width=0.55)
    ax.set_ylabel("Engineer's view: wrong-and-undeferred (%)", color="black")
    ax.set_ylim(0, 105)
    ax.set_xticks(x)
    ax.set_xticklabels(conditions)
    ax.set_title(mechanism.replace("_", " ").title())
    for bar, v in zip(bars, wu):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

    ax2.plot(x, defr, color="black", marker="o", linewidth=2, label="Person's view: deferral rate")
    ax2.set_ylabel("Person's view: deferral rate (%)", color="black")
    ax2.set_ylim(0, 105)
    for xi, v in zip(x, defr):
        ax2.text(xi, v + 3, f"{v:.1f}%", ha="center", fontsize=9, color="black")

    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "Engineer's view (bars) vs. person's view (line) across conditions",
    fontsize=13, y=1.02,
)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "asymmetry_per_mechanism.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'asymmetry_per_mechanism.png'}")


In [ ]:
# -------------------------------------------------------------
# Part K: Figure — Inversion magnitude bar chart
# -------------------------------------------------------------

inv = asym_df[asym_df["condition"] == "adaptive"].copy()

fig, ax = plt.subplots(figsize=(8.5, 5))
x = np.arange(len(inv))
width = 0.35

bars_a = ax.bar(x - width/2, inv["delta_deferral_pp"], width,
                label="Δdeferral (clean − adaptive, pp)", color="#4C78A8")
bars_b = ax.bar(x + width/2, inv["delta_exposure_pp"], width,
                label="Δexposure (adaptive − clean, pp)", color="#E45756")

ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", " ").title() for m in inv["mechanism"]])
ax.set_ylabel("Percentage points")
ax.set_title("Inversion test under adaptive attack\n(both bars positive = signal points opposite to reality)")
ax.legend(loc="best", fontsize=9)
ax.grid(axis="y", alpha=0.25)

for bars in (bars_a, bars_b):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, h + (1 if h >= 0 else -1),
                f"{h:+.1f}", ha="center", va="bottom" if h >= 0 else "top", fontsize=9)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "inversion_adaptive.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'inversion_adaptive.png'}")


In [ ]:
# -------------------------------------------------------------
# Part L: Budget Sweep — repeat at 1/255, 2/255, 4/255
# -------------------------------------------------------------
#
# At 8/255 all three classifiers reach 0% accuracy under direct
# attack, so the adaptive condition can hit a ceiling. The sweep
# below repeats direct and adaptive at smaller budgets, using the
# lambdas selected in #10's sweep (if present), to see whether the
# inversion appears where the ceiling does not apply.
# -------------------------------------------------------------

SWEEP_EPSILONS = [1, 2, 4, 8]
SWEEP_ROWS = []

SWEEP_LAMBDAS_PATH = Path("essay10_results_corrected/epsilon_sweep_gaps.csv")
if SWEEP_LAMBDAS_PATH.exists():
    sweep_ref = pd.read_csv(SWEEP_LAMBDAS_PATH)
else:
    sweep_ref = None
    print("Note: #10's budget-sweep lambdas not found. "
          "Adaptive condition will use the main-run lambda for all budgets.")

for mechanism, model in models.items():
    thr = CALIBRATION[mechanism]["threshold"]
    scale = CALIBRATION[mechanism]["scale"]

    for e in SWEEP_EPSILONS:
        eps = e / 255.0
        # Prefer the lambda #10 tuned for this exact budget
        if sweep_ref is not None:
            row = sweep_ref[(sweep_ref["mechanism"] == mechanism)
                            & (sweep_ref["epsilon_255"] == e)]
            lam = float(row["selected_lambda"].iloc[0]) if not row.empty else SELECTED_LAMBDAS[mechanism]
        else:
            lam = SELECTED_LAMBDAS[mechanism]

        print(f"\n{mechanism} @ {e}/255 | lambda={lam:.4f}")

        for condition in ("clean", "direct", "adaptive"):
            r = evaluate_condition(
                mechanism, model, eval_loader,
                threshold=thr, condition=condition,
                adaptive_lambda=lam if condition == "adaptive" else None,
                uncertainty_scale=scale if condition == "adaptive" else None,
                epsilon=eps,
            )
            SWEEP_ROWS.append({"mechanism": mechanism, "epsilon_255": e,
                               "condition": condition, "lambda": lam, **r})
            print(f"  {condition:<9} acc={r['accuracy']:6.2f}%  def={r['deferral_rate']:6.2f}%  WU={r['wrong_undeferred']:6.2f}%")

sweep_df = pd.DataFrame(SWEEP_ROWS)
sweep_df.to_csv(RESULTS_DIR / "sweep.csv", index=False)


In [ ]:
# -------------------------------------------------------------
# Part M: Inversion across budgets
# -------------------------------------------------------------

sweep_inv_rows = []
for mechanism in models:
    for e in SWEEP_EPSILONS:
        sub = sweep_df[(sweep_df["mechanism"] == mechanism)
                       & (sweep_df["epsilon_255"] == e)].set_index("condition")
        if not {"clean", "direct", "adaptive"} <= set(sub.index):
            continue
        clean = sub.loc["clean"]
        for condition in ("direct", "adaptive"):
            a = sub.loc[condition]
            bin_inv = inversion_binary(clean["deferral_rate"], clean["wrong_undeferred"],
                                       a["deferral_rate"], a["wrong_undeferred"])
            mag, d_def, d_exp = inversion_magnitude(clean["deferral_rate"], clean["wrong_undeferred"],
                                                    a["deferral_rate"], a[")
                                                                          